# Bao 優先相転移候補・局面前後監査

A群の主要6アーキタイプについて、代表局面の直前1手から直後8手までを再構成し、盤面・手・特徴量・勝敗を確認する。


## 1. 入力ZIPを展開


In [ ]:
from pathlib import Path
import zipfile

ZIP_PATH = Path('/content/pilot-v2-analysis-input.zip')
PILOT_V2 = Path('/content/pilot-v2')
PILOT_V2.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH) as archive:
    archive.extractall(PILOT_V2)
for name in ['observations.jsonl', 'games.json', 'manifest.json']:
    path = PILOT_V2 / name
    assert path.exists(), f'Missing: {path}'
    print(name, path.stat().st_size)


## 2. 最新リポジトリでアーキタイプと局面監査を実行


In [ ]:
REPO = Path('/content/bao-la-kiswahili-game')
ARCHETYPE_OUTPUT = Path('/content/phase-transition-archetypes')
AUDIT_OUTPUT = Path('/content/phase-transition-candidate-audit')
!rm -rf {REPO} {ARCHETYPE_OUTPUT} {AUDIT_OUTPUT}
!git clone -q https://github.com/nkkmd/bao-la-kiswahili-game.git {REPO}
!git -C {REPO} log -1 --oneline
!python {REPO}/tools/experiments/analyze-phase-transition-archetypes.py --input {PILOT_V2} --output {ARCHETYPE_OUTPUT}
!node {REPO}/tools/experiments/extract-phase-transition-candidate-audit.js --input {PILOT_V2} --archetypes {ARCHETYPE_OUTPUT}/candidate-archetypes.csv --output {AUDIT_OUTPUT}


## 3. 監査対象と時系列を表示


In [ ]:
import json
import pandas as pd

summary = json.loads((AUDIT_OUTPUT / 'candidate-board-audit-summary.json').read_text(encoding='utf-8'))
audit = pd.read_csv(AUDIT_OUTPUT / 'candidate-board-audit.csv')
print(summary)
display(audit[['archetypeId','gameId','targetPly','ply','relativePly','phase','player','forcedCapture','legalMoveCount','captureMoveCount','selectedMove','winner']])


## 4. 盤面を表示


In [ ]:
def board_frame(row):
    return pd.DataFrame(
        [
            json.loads(row['northBack']),
            json.loads(row['northFront']),
            json.loads(row['southFront']),
            json.loads(row['southBack']),
        ],
        index=['North back ←', 'North front ←', 'South front →', 'South back →'],
        columns=[str(i) for i in range(8)],
    )

for archetype_id in summary['selectedArchetypeIds']:
    subset = audit[audit['archetypeId'] == archetype_id]
    target = subset[subset['isTarget'] == True].iloc[0]
    print('\n', '=' * 80)
    print(
        f"Archetype {archetype_id} | {target['gameId']} | target ply {int(target['targetPly'])} | "
        f"phase={target['phase']} | player={int(target['player'])}"
    )
    for relative in [-1, 0, 1, 3, 5, 8]:
        rows = subset[subset['relativePly'] == relative]
        if rows.empty:
            continue
        row = rows.iloc[0]
        print(
            f"\nrelative={relative:+d}, ply={int(row['ply'])}, "
            f"legal={int(row['legalMoveCount'])}, captures={int(row['captureMoveCount'])}, "
            f"move={row['selectedMove']}"
        )
        display(board_frame(row))


## 5. Google Driveへ保存


In [ ]:
from google.colab import drive
import shutil

drive.mount('/content/drive')
DESTINATION = Path('/content/drive/MyDrive/bao-la-kiswahili-game/phase-transition-analysis/pilot-v2-candidate-board-audit')
DESTINATION.mkdir(parents=True, exist_ok=True)
for filename in [
    'candidate-board-audit-summary.json',
    'candidate-board-audit.json',
    'candidate-board-audit.csv',
]:
    shutil.copy2(AUDIT_OUTPUT / filename, DESTINATION / filename)
for path in sorted(DESTINATION.iterdir()):
    print(path.name, path.stat().st_size)
